# Sampling Test

## Information

Técnicas de muestreo del módulo `scipy.stats.qmc`:
* Sobol: <--
    * Permite ampliar progresivamente el número de muestras
    * **Se recomienda que el número total de muestras sea una potencia de 2**.
* Latin Hypercube (LHS)
    * Funciona con muestras bajas
    * **NO** ampliar el número de muestras


<font color=fa5>**Nota**</font>:
Las siguientes funciones se han movido a AUX (el cuaderno sigue usando las funciones locales):
*  `aux.parameters` $\longleftarrow$ `flatten_params`, `restore_flat_params`, `add_truncnorm_config`, `add_beta_config`. 
*  `aux.sampling` $\longleftarrow$ `create_sobol`, `_uniform_sample`, `_truncnorm_sample`, `_beta_sample`, `_sample_from_pdf`, `sample_params`, `sample_weight`.

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import numpy as np
from scipy.stats import qmc
from scipy.stats import beta, truncnorm, uniform
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from ipywidgets import FloatSlider, interact, ToggleButton

from copy import deepcopy

In [ ]:
from aux.environment import resolve_directory

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:

WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Basic Sobol Sampling

In [ ]:
N_VARIABLES = 30
M_SAMPLES = 2**10
SEED = 42

In [ ]:
def create_sobol():
    return qmc.Sobol(
        d=N_VARIABLES, scramble=True, seed=SEED
    )


In [ ]:
sampler = create_sobol()

m = int(np.log2(M_SAMPLES))-1
batch11 = sampler.random_base2(m=m)
batch12 = sampler.random_base2(m=m)
batch1 = np.vstack((batch11, batch12))    

pprint(f'Batch shape: {batch1.shape}, Batch size: {batch1.nbytes / 1024**2:.2f} MB')

In [ ]:
sampler = create_sobol()

# Puedes trabajar sin la restricción de que sea una potencia de 2, **pero no es recomendable**.
batch2 = np.vstack([sampler.random(M_SAMPLES // 4) for _ in range(4)])

pprint(f'Batch shape: {batch2.shape}, Batch size: {batch2.nbytes / 1024**2:.2f} MB')

In [ ]:
# Test para verificar como aplicar el número de muestras en distintos ficheros.

batch3_parts = []
for i in range(4):
    sampler = create_sobol()
    if i != 0: 
        sampler.fast_forward(i * (M_SAMPLES // 4))
    batch3_parts.append(sampler.random(M_SAMPLES // 4))

batch3 = np.vstack(batch3_parts)

In [ ]:
## Proyección PCA de los tres lotes
pca = PCA(n_components=2).fit(batch1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)

for ax, (name, batch) in zip(axes, [('Batch 1', batch1), ('Batch 2', batch2), ('Batch 3', batch3)]):
    projected = pca.transform(batch)
    ax.scatter(projected[:, 0], projected[:, 1], s=1)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])


fig.suptitle('Sobol Sampling PCA Projection')
fig.tight_layout()
plt.show()

## Distributions

In [ ]:
MIN = 0.0
MAX = 1.0
PEAK = 0.2*MAX
LAMBDA = 3 
STD = 0.3*(MAX-MIN)

In [ ]:
relative_peak = (PEAK - MIN) / (MAX - MIN)

beta_a = 1 + relative_peak * LAMBDA
beta_b = 1 + (1 - relative_peak) * LAMBDA

# Truncated normal distribution
trunc_a = (MIN - PEAK) / STD
trunc_b = (MAX - PEAK) / STD

distributions = [
    ("Asymmetric Beta", beta(beta_a, beta_b, loc=MIN, scale=MAX - MIN)),
    ("Truncated Normal", truncnorm(trunc_a, trunc_b, loc=PEAK, scale=STD)),
    ("Uniform", uniform(loc=MIN, scale=MAX - MIN)),
]

In [ ]:
x = np.linspace(MIN, MAX, 1000)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for ax, (title, dist) in zip(axes.flat, distributions):
    y = dist.pdf(x)
    ax.plot(x, y)
    ax.fill_between(x, y, alpha=0.2)
    ax.set_title(title)
    ax.set_xlim(MIN, MAX)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Parameter sampling

In [ ]:
PARAMS = {
    "V1": { "MIN": 10, "MAX": 20, "DEFAULT": 15 },
    "V2": { "MIN": 0.1, "MAX": 1.0, "DEFAULT": 0.5 },
}

PARAMS2 = {
    "V1": { "MIN": 10, "MAX": 20, "DEFAULT": 15 },
    "V2": { "MIN": [0.2, 0.4], "MAX": [0.8, 1.0], "DEFAULT": [0.5, 0.7] },
}

In [ ]:
def flatten_params(params):
    flat = {}
    for name, config in params.items():
        first = next(iter(config.values()))
        if isinstance(first, list):
            for i in range(len(first)):
                flat[f"{name}*{i:02}"] = {k: v[i] for k, v in config.items()}
        else:
            flat[name] = config
    return flat


def restore_flat_params(flat_params):
    result = {}
    for name, config in sorted(flat_params.items()):
        base, sep, _ = name.partition("*")
        if not sep:
            result[name] = config
        else:
            group = result.setdefault(base, {})
            for k, v in config.items():
                group.setdefault(k, []).append(v)
    return result

flattened_params = flatten_params(PARAMS2)
pprint('Flattened PARAMS2:', flattened_params)
pprint('Restored sample:', restore_flat_params(flattened_params))


In [ ]:
def add_truncnorm_config(params, std_factor=0.2):
    result =deepcopy(params)

    for name in result.keys():
        result[name]["STD"]= (result[name]["MAX"] - result[name]["MIN"]) * std_factor
        result[name]["METHOD"] = "truncnorm"

    return result

def add_beta_config(params, lambda_):
    result =deepcopy(params)

    for name in result.keys():
        result[name]["LAMBDA"] = lambda_
        result[name]["METHOD"] = "beta"

    return result

beta_params = add_beta_config(PARAMS, lambda_=5)
truncnorm_params = add_truncnorm_config(PARAMS, std_factor=0.2)

pprint('Beta parameters:', beta_params)
pprint('Truncated Normal parameters:', truncnorm_params)

In [ ]:
def _uniform_sample(s, minimum, maximum):
    scale= maximum - minimum
    return minimum + s * scale, 1/scale


def _truncnorm_sample(s, minimum, maximum, peak, std):
    a = (minimum - peak) / std
    b = (maximum - peak) / std
    dist = truncnorm(a, b, loc=peak, scale=std)
    return (value := dist.ppf(s)), dist.pdf(value)


def _beta_sample(s, minimum, maximum, peak, lam):
    relative_peak = (peak - minimum) / (maximum - minimum)
    a = 1 + relative_peak * lam
    b = 1 + (1 - relative_peak) * lam
    dist = beta(a, b, loc=minimum, scale=maximum - minimum)
    return (value := dist.ppf(s)), dist.pdf(value)

def _sample_from_pdf(s, param):
    minimum = param["MIN"]
    maximum = param["MAX"]

    match param.get("METHOD", "uniform"):
        case "truncnorm":
            return _truncnorm_sample(
                s, minimum, maximum, param["DEFAULT"], param["STD"]
            )
        case "beta":
            return _beta_sample(
                s, minimum, maximum, param["DEFAULT"], param["LAMBDA"]
            )
        case _:
            return _uniform_sample(s, minimum, maximum)


def sample_params(params, n_samples, fordware=0, seed=42):
    sampler = qmc.Sobol(d=len(params), scramble=True, seed=seed)
    if fordware > 0:
        sampler.fast_forward(fordware)

    for s in sampler.random(n_samples):
        values, densities = {}, {}
        for i, (name, config) in enumerate(params.items()):
            value, density = _sample_from_pdf(s[i], config)
            values[name] = value
            densities[name] = density

        yield values, densities

        # # Mas elegante, pero menos eficiente
        # values, densities = zip(*map(_sample_from_pdf, s, params.values()))
        # yield dict(zip(params, values)), dict(zip(params, densities))



points, pdfs = map(list, zip(*sample_params(PARAMS, n_samples=1024, fordware=0, seed=42)))
pprint(f'Sampled {len(points)}')

In [ ]:
configs = [
    ('Truncada, σ = 0.1 × rango', add_truncnorm_config(PARAMS, std_factor=0.1)),
    ('Truncada, σ = 0.3 × rango', add_truncnorm_config(PARAMS, std_factor=0.3)),
    ('Beta, λ = 1', add_beta_config(PARAMS, lambda_=1)),
    ('Beta, λ = 1.5', add_beta_config(PARAMS, lambda_=1.5)),
    ('Beta, λ = 3', add_beta_config(PARAMS, lambda_=3)),
    ('Uniform', PARAMS)
]
x_name, y_name = list(PARAMS)[:2]

fig, axes = plt.subplots(2, 3, figsize=(12, 4), sharex=True, sharey=True, constrained_layout=True)

for ax, (title, config) in zip(axes.T.flat, configs):
    points, pdfs = map(list, zip(*sample_params(config, n_samples=1024)))
    scatter = ax.scatter(
        [p[x_name] for p in points], [p[y_name] for p in points],
        c=[p[x_name] * p[y_name] for p in pdfs], cmap='viridis'
    )
    ax.set(xlabel=x_name, ylabel=y_name, title=title)
    fig.colorbar(scatter, ax=ax)

plt.show()

## Weights


Si se usan distribuciones de probabilidad al rededor de la media para el muestreo, así garantizamos que el modelo aprenda bien esa zona. No obstante, podemos aumentar la ponderación de las zonas poco probables para que apesar de tener pocas muestras, tengan un impacto en el entrenamiento del modelo.

**Pdf usada en el muestreo**

$$
Q(\mathbf{X}) = \prod_{i=1}^{n} q_i(x_i)
$$

**Si suponemos que la distribución de probabilidad de cada parámetro es uniforme y que las variables son independientes**, la distribucion real de la probabilidad puede calcularse como:

$$
\begin{align}
U(\mathbf{X}) = \prod_{i=1}^{n} u_i(x_i) \\
u_i(x_i) = cte \quad \forall x_i \\
U(\mathbf{X}) = cte \quad \forall \mathbf{X}
\end{align}
$$ 

**Las ténicas de simulación de Monte Carlo calculan un peso para cada muestra**. Este peso es el cociente entre la distribución real de probabilidad y la distribución usada en el muestreo:

$$
W(\mathbf{X}) = \frac{U(\mathbf{X})}{Q(\mathbf{X})}
$$

En DL, como el la distribución de probablidad es constante:

$$
W(\mathbf{X}) = \frac{U(\mathbf{X})}{Q(\mathbf{X})}
\overset{\mathrm{optimization}}{\equiv}
\frac{1}{Q(\mathbf{X})}
$$

En nuestro modelo DL, (i) no sabemos cual es la distribución real (puede que esta no este tan alejada de la de muestreo), (ii) puede que dividir por la distribución de muestreo de demasiado peso a valores extremos. Lo podemos solucionar considernado que la distribución real es una distribución que se obtiene de interpolar la distribución de muestreo con una distribución uniforme. 


$$
P(\mathbf{X}) = (1 - \alpha) Q(\mathbf{X}) + \alpha U(\mathbf{X})
$$

De esta forma el peso quedaría de la siguiente forma:

$$
W(\mathbf{X}) = (1-\alpha) + \alpha \frac{U(\mathbf{X})}{Q(\mathbf{X})} 
\overset{\mathrm{optimization}}{\equiv} 
(1-\alpha) + \alpha \frac{1}{Q(\mathbf{X})}
$$




### Dos parámetros, multiples distribuciones

In [ ]:
PARAMS = {
    'V1': {'MIN': 0, 'MAX': 1, 'DEFAULT': 0.25},
    'V2': {'MIN': 2, 'MAX': 10, 'DEFAULT': 7.5},
}

sample_configs = [
    ('Truncada, σ=0.2', add_truncnorm_config(PARAMS, 0.2)),
    ('Truncada, σ=0.35', add_truncnorm_config(PARAMS, 0.35)),
    ('Beta, λ=3', add_beta_config(PARAMS, 3)),
    ('Uniforme', PARAMS),
]

sample_set = [(name, list(sample_params(config, 512))) for name, config in sample_configs]

In [ ]:
def sample_weight(pdfs, alpha=0.5):
    return (1 - alpha) + alpha / np.prod(list(pdfs.values()))

def weight_norm(weights, lognorm):
    if not lognorm:
        return None
    lo, hi = min(weights), max(weights)
    if lo == hi:
        lo, hi = lo / 1.1, hi * 1.1
    return LogNorm(vmin=lo, vmax=hi)

In [ ]:

@interact(alpha=FloatSlider(value=0.5, min=0, max=1, step=0.05,
                             description='α', continuous_update=False),
          lognorm=ToggleButton(value=False, description='LogNorm'))
def plot_weights(alpha, lognorm):
    weights = [[sample_weight(pdf, alpha) for _, pdf in samples]
               for _, samples in sample_set]
    
    all_weights = np.concatenate(weights)

    pprint(f'Weight shape: {all_weights.shape}')

    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True,
                             sharey=True, constrained_layout=True)
    for ax, (name, samples), color in zip(axes.flat, sample_set, weights):
        points = [point for point, _ in samples]
        plot = ax.scatter([p['V1'] for p in points],
                          [p['V2'] for p in points], c=color,
                          norm=weight_norm(color, lognorm),
                          cmap='viridis', s=10)
        ax.set(xlabel='V1', ylabel='V2', title=name)
        fig.colorbar(plot, ax=ax, label='Peso W')
    plt.show()

### Beta, 10 parámetros, 1024 muestras y proyección PCA

In [ ]:
rng = np.random.default_rng(42)
mins = rng.uniform(0, 7, 10)
maxs = mins + rng.uniform(1, 3, 10)
relative_peaks = rng.uniform(0.2, 0.8, 10)

PARAMS_BETA10 = {
    f'V{i + 1}': {'MIN': lo, 'MAX': hi,
                  'DEFAULT': lo + relative_peak * (hi - lo)}
    for i, (lo, hi, relative_peak) in enumerate(
        zip(mins, maxs, relative_peaks))
}

beta10_samples = list(sample_params(add_beta_config(PARAMS_BETA10, 3),
                                    n_samples=1024))
beta10_values = np.array([[values[name] for name in PARAMS_BETA10]
                          for values, _ in beta10_samples])
beta10_pca = PCA(n_components=2)
beta10_xy = beta10_pca.fit_transform(beta10_values)

@interact(alpha=FloatSlider(value=0.5, min=0, max=1, step=0.05,
                             description='α', continuous_update=False),
          lognorm=ToggleButton(value=False, description='LogNorm'))
def plot_beta10_pca(alpha, lognorm):
    weights = [sample_weight(pdfs, alpha) for _, pdfs in beta10_samples]
    fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
    plot = ax.scatter(beta10_xy[:, 0], beta10_xy[:, 1], c=weights,
                      norm=weight_norm(weights, lognorm),
                      cmap='viridis', s=12)
    ax.set_title('Beta: 10 parámetros, 1024 muestras')
    fig.colorbar(plot, ax=ax, label='Peso W')
    plt.show()